# Sign Language Classifier — Training on Colab (with W&B)

**Capstone CV 2025.2** — Trần Quang Hưng, Đỗ Đăng Vũ, Lê Hoàng Tùng.

Notebook tự đứng độc lập (không cần upload code khác). Train **ResNet18** classifier 29 lớp ASL trên Kaggle ASL Alphabet với log đầy đủ qua **Weights & Biases**.

## Yêu cầu trước khi chạy

1. **GPU**: Runtime → Change runtime type → **T4 GPU**.
2. **Kaggle API key** (`kaggle.json`): https://www.kaggle.com/settings → *Create New API Token*.
3. **W&B API key**: https://wandb.ai/authorize (đăng ký free nếu chưa có).
4. (Tùy chọn) Mount Google Drive để lưu checkpoint.

## W&B sẽ log những gì

- **Config**: tất cả hyperparameters
- **Scalars** (per epoch): train_loss, val_loss, val_acc, learning_rate, epoch_time
- **System metrics**: GPU util, GPU memory, CPU
- **Sample predictions** mỗi epoch (16 ảnh val + label thật + label dự đoán)
- **Confusion matrix** cuối training
- **Top-5 confused class pairs** (vd M↔N, R↔U)
- **Model artifact** (`cnn_resnet18.pt`) versioned theo run

Thời gian: ~30 phút trên T4 cho 10 epochs.

## 0 · Kiểm tra GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    !nvidia-smi -L
else:
    print('⚠️  Không thấy GPU — bật trong Runtime → Change runtime type.')

## 1 · Mount Google Drive (tùy chọn)

In [ ]:
import os
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/signlang'
else:
    SAVE_DIR = '/content/signlang_out'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Output dir: {SAVE_DIR}')

## 2 · Cài deps + login W&B

In [ ]:
!pip install -q kaggle wandb scikit-learn

import wandb
wandb.login()   # paste W&B API key (lấy ở https://wandb.ai/authorize)

## 3 · Tải Kaggle ASL Alphabet

In [ ]:
from google.colab import files
import os, shutil

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload kaggle.json:')
    files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

if not os.path.exists('/content/data/asl_alphabet_train/asl_alphabet_train/A'):
    !kaggle datasets download -d grassknoted/asl-alphabet -p /content/data
    !unzip -q /content/data/asl-alphabet.zip -d /content/data/

DATA_DIR = '/content/data/asl_alphabet_train/asl_alphabet_train'
TEST_DIR = '/content/data/asl_alphabet_test/asl_alphabet_test'
print(f'Classes: {len(os.listdir(DATA_DIR))}, samples in A: {len(os.listdir(os.path.join(DATA_DIR, "A")))}')

## 4 · Hyperparameters + dataset + model

Đây là cell duy nhất bạn cần sửa nếu muốn ablation (ResNet18 vs MobileNetV2, batch size khác, ...). W&B sẽ tự log toàn bộ config dictionary.

In [ ]:
config = {
    'arch': 'resnet18',                # 'resnet18' | 'mobilenet_v2'
    'pretrained': True,
    'image_size': 224,
    'batch_size': 64,
    'epochs': 10,
    'lr': 1e-3,
    'weight_decay': 5e-4,
    'optimizer': 'AdamW',
    'scheduler': 'CosineAnnealingLR',
    'augment_rotation_deg': 15,
    'augment_color_jitter': 0.2,
    'val_split': 0.15,
    'seed': 11711,
    'num_classes': 29,
    'amp': True,
    'dataset': 'kaggle/asl-alphabet',
}

CLASS_NAMES = ['A','B','C','D','E','F','G','H','I','J','K','L','M',
               'N','O','P','Q','R','S','T','U','V','W','X','Y','Z',
               'space','del','nothing']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights, mobilenet_v2, MobileNet_V2_Weights
from torchvision.transforms import v2 as T

train_tf = T.Compose([
    T.Resize((config['image_size'], config['image_size']), antialias=True),
    T.RandomRotation(config['augment_rotation_deg']),
    T.ColorJitter(config['augment_color_jitter'], config['augment_color_jitter'], config['augment_color_jitter']),
    T.ToImage(), T.ToDtype(torch.float32, scale=True),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = T.Compose([
    T.Resize((config['image_size'], config['image_size']), antialias=True),
    T.ToImage(), T.ToDtype(torch.float32, scale=True),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class RemappedFolder(Dataset):
    def __init__(self, root, transform):
        self.inner = ImageFolder(root, transform=transform)
        canonical = {n: i for i, n in enumerate(CLASS_NAMES)}
        self.remap = [canonical[n] for n in self.inner.classes]
    def __len__(self): return len(self.inner)
    def __getitem__(self, idx):
        x, y = self.inner[idx]
        return x, self.remap[y]

full_train = RemappedFolder(DATA_DIR, train_tf)
val_size = int(config['val_split'] * len(full_train))
train_size = len(full_train) - val_size
g = torch.Generator().manual_seed(config['seed'])
train_ds, val_ds = random_split(full_train, [train_size, val_size], generator=g)
val_ds.dataset = RemappedFolder(DATA_DIR, eval_tf)

train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True,
                          num_workers=2, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=config['batch_size'], shuffle=False,
                        num_workers=2, pin_memory=True, persistent_workers=True)

def build_model(cfg):
    if cfg['arch'] == 'resnet18':
        m = resnet18(weights=ResNet18_Weights.DEFAULT if cfg['pretrained'] else None)
        m.fc = nn.Linear(m.fc.in_features, cfg['num_classes'])
    elif cfg['arch'] == 'mobilenet_v2':
        m = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT if cfg['pretrained'] else None)
        m.classifier = nn.Sequential(nn.Dropout(0.2),
                                     nn.Linear(m.classifier[-1].in_features, cfg['num_classes']))
    else:
        raise ValueError(cfg['arch'])
    return m

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Device: {DEVICE}')

## 5 · Train với W&B logging

Logs đầy đủ: scalars + sample predictions + system metrics.

In [ ]:
import time
import numpy as np
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm

# Init W&B run
run = wandb.init(
    project='signlang-classifier',
    name=f"{config['arch']}-bs{config['batch_size']}-lr{config['lr']}",
    config=config,
    tags=['classifier', config['arch']],
    notes='ASL alphabet 29-class CNN classifier',
)

model = build_model(config).to(DEVICE)
wandb.watch(model, log='gradients', log_freq=100)   # auto-log gradients + params histograms
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
wandb.log({'model/trainable_params': trainable, 'model/total_params': sum(p.numel() for p in model.parameters())})

optimizer = AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=config['epochs'])
scaler = torch.amp.GradScaler('cuda') if (config['amp'] and DEVICE.type == 'cuda') else None

best_acc = 0.0
ckpt_path = '/content/cnn_classifier.pt'

# Pre-fetch a fixed val batch for sample-prediction logging
vis_batch = next(iter(val_loader))
vis_x, vis_y = vis_batch[0][:16].to(DEVICE), vis_batch[1][:16]
vis_imgs_unnorm = (vis_batch[0][:16] * torch.tensor(IMAGENET_STD).view(3,1,1)
                   + torch.tensor(IMAGENET_MEAN).view(3,1,1)).clamp(0, 1)

for epoch in range(config['epochs']):
    t0 = time.time()
    # ---- Train ----
    model.train()
    train_loss_sum = 0.0; n = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']}")
    for x, y in pbar:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(x); loss = nn.functional.cross_entropy(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        else:
            logits = model(x); loss = nn.functional.cross_entropy(logits, y)
            loss.backward(); optimizer.step()
        train_loss_sum += loss.item() * x.size(0); n += x.size(0)
        pbar.set_postfix(loss=f'{train_loss_sum/n:.4f}')
    train_loss = train_loss_sum / n

    # ---- Val ----
    model.eval()
    val_loss_sum = 0.0; correct = 0; total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE); y = y.to(DEVICE)
            logits = model(x); loss = nn.functional.cross_entropy(logits, y)
            val_loss_sum += loss.item() * x.size(0)
            correct += (logits.argmax(-1) == y).sum().item()
            total += y.size(0)
    val_loss = val_loss_sum / total; val_acc = correct / total
    scheduler.step()
    epoch_time = time.time() - t0

    # ---- W&B log ----
    with torch.no_grad():
        vis_logits = model(vis_x)
        vis_preds = vis_logits.argmax(-1).cpu().numpy()
    sample_predictions = [
        wandb.Image(
            vis_imgs_unnorm[i].permute(1, 2, 0).numpy(),
            caption=f'true={CLASS_NAMES[int(vis_y[i])]} | pred={CLASS_NAMES[int(vis_preds[i])]}',
        )
        for i in range(min(16, len(vis_y)))
    ]

    wandb.log({
        'epoch': epoch + 1,
        'train/loss': train_loss,
        'val/loss': val_loss,
        'val/accuracy': val_acc,
        'lr': scheduler.get_last_lr()[0],
        'time/epoch_seconds': epoch_time,
        'samples/predictions': sample_predictions,
    })
    print(f'Epoch {epoch+1}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  ({epoch_time:.0f}s)')

    # ---- Save best ----
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'model': model.state_dict(),
            'arch': config['arch'],
            'num_classes': config['num_classes'],
            'class_names': CLASS_NAMES,
            'image_size': config['image_size'],
            'val_acc': val_acc,
            'wandb_run_id': run.id,
        }, ckpt_path)
        print(f'  ✓ New best — saved (acc={val_acc:.4f})')

print(f'\nBest val accuracy: {best_acc:.4f}')
wandb.summary['best_val_accuracy'] = best_acc

## 6 · Confusion matrix + top confused pairs

Phân tích lỗi sau cùng — log lên W&B để dùng trong báo cáo.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Load best checkpoint
ckpt = torch.load(ckpt_path, weights_only=False)
model.load_state_dict(ckpt['model'])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        all_preds.extend(model(x).argmax(-1).cpu().numpy())
        all_labels.extend(y.numpy())

all_preds = np.array(all_preds); all_labels = np.array(all_labels)
cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(CLASS_NAMES))))

# 1) Log confusion matrix as W&B plot
wandb.log({'val/confusion_matrix': wandb.plot.confusion_matrix(
    probs=None, y_true=all_labels.tolist(), preds=all_preds.tolist(),
    class_names=CLASS_NAMES,
)})

# 2) Top-10 confused pairs (off-diagonal cells)
off_diag = cm.copy(); np.fill_diagonal(off_diag, 0)
flat_idx = off_diag.flatten().argsort()[::-1][:10]
confused_table = wandb.Table(columns=['true', 'predicted', 'count', '% of true class'])
for idx in flat_idx:
    i, j = divmod(idx, len(CLASS_NAMES))
    if cm[i, j] == 0: continue
    confused_table.add_data(CLASS_NAMES[i], CLASS_NAMES[j], int(cm[i, j]),
                            f'{cm[i,j] / cm[i].sum() * 100:.1f}%')
wandb.log({'val/top_confused_pairs': confused_table})

# 3) Classification report (per-class precision/recall/f1)
report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES,
                                output_dict=True, zero_division=0)
report_table = wandb.Table(columns=['class', 'precision', 'recall', 'f1', 'support'])
for cname in CLASS_NAMES:
    r = report[cname]
    report_table.add_data(cname, r['precision'], r['recall'], r['f1-score'], r['support'])
wandb.log({'val/per_class_metrics': report_table})

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0))

## 7 · Đánh giá Kaggle test set + lưu artifact

In [ ]:
import glob, re
from PIL import Image

test_paths = sorted(glob.glob(f'{TEST_DIR}/*_test.jpg'))
test_correct = 0; test_total = 0
test_table = wandb.Table(columns=['image', 'true', 'predicted', 'top1_conf'])
for p in test_paths:
    m = re.match(r'(.+)_test\.jpg', os.path.basename(p))
    if not m: continue
    name = m.group(1)
    if name not in CLASS_NAMES: continue
    truth = CLASS_NAMES.index(name)
    img = Image.open(p).convert('RGB')
    x = eval_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = model(x).softmax(-1)[0].cpu().numpy()
    pred = int(probs.argmax())
    test_table.add_data(wandb.Image(img), name, CLASS_NAMES[pred], float(probs[pred]))
    test_correct += int(pred == truth); test_total += 1

test_acc = test_correct / max(1, test_total)
wandb.summary['kaggle_test_accuracy'] = test_acc
wandb.log({'test/predictions': test_table, 'test/accuracy': test_acc})
print(f'Kaggle test accuracy: {test_correct}/{test_total} = {test_acc*100:.1f}%')

# Save model artifact (versioned in W&B — bạn có thể download từ web UI)
artifact = wandb.Artifact(name=f"{config['arch']}-asl-classifier", type='model',
                          description=f"Best val_acc={best_acc:.4f} on Kaggle ASL alphabet",
                          metadata={'best_val_acc': best_acc, 'kaggle_test_acc': test_acc})
artifact.add_file(ckpt_path)
run.log_artifact(artifact)

# Copy to Drive
import shutil
shutil.copy(ckpt_path, f'{SAVE_DIR}/cnn_{config["arch"]}.pt')
print(f'\nSaved to:\n  Drive: {SAVE_DIR}/cnn_{config["arch"]}.pt\n  W&B artifact: {artifact.name}')

wandb.finish()

## Bước tiếp theo

1. Mở W&B dashboard: https://wandb.ai/your-username/signlang-classifier — review charts, confused pairs, sample predictions.
2. Train YOLO detector: chạy notebook `colab_train_yolo_detector.ipynb`.
3. Tải `cnn_resnet18.pt` về máy + chạy demo: `make demo` (cần webcam).

## Ablation experiments (nếu rảnh)

Ở cell hyperparameters (cell 4), đổi `arch: 'mobilenet_v2'` rồi chạy lại notebook → W&B tự tạo run mới. So sánh 2 runs trên dashboard để có data ablation cho báo cáo.